## Task 1: Tensors & Autograd

Learn how PyTorch represents data.

Practice creating tensors, doing basic operations, and enabling gradients.

Example goals:

Create a
4
×
3
4×3 weight matrix and multiply it with a
4
4-dim input.

Backpropagate a simple loss (like sum of outputs).

### 👉 Concept learned: Tensors, shapes, autograd.

In [ ]:
import torch
import torch.nn as nn

In [ ]:
W = torch.rand([4,3],requires_grad=True)
x = torch.rand(4)
y = torch.rand(3)
b = torch.rand(3,requires_grad=True)
y_ = x@W+b

mse_loss = nn.MSELoss()

loss = mse_loss(y,y_)
loss.backward()

## Task 2: Simple Linear Layer from Scratch

Implement a single fully connected layer (like we just discussed) without using nn.Linear.

Use torch.mm for matrix multiplication and manually apply weights + bias.

Train it on toy data (e.g., learn mapping
𝑦
=
2
𝑥
+
1
y=2x+1).

```
This code creates two tensors, x and y, which represent the input and output data for a simple linear regression problem.

x = torch.linspace(-5, 5, 100).unsqueeze(1): This line creates a tensor x containing 100 evenly spaced values between -5 and 5. unsqueeze(1) adds an extra dimension to the tensor, changing its shape from (100,) to (100, 1). This is often done to prepare the data for matrix multiplication in neural networks.
y = 2 * x + 1: This line creates a tensor y by applying the linear function y = 2x + 1 to each element in the x tensor. This represents the target values that a model would try to predict based on the input x.
```

In [ ]:
x = torch.linspace(-5, 5, 100).unsqueeze(1)   # shape (100, 1)
y = 4 * x + 3
W = torch.randn(1, requires_grad=True)  # scalar weight
b = torch.randn(1, requires_grad=True)  # scalar bias

def forward(x):
    return x * W + b

loss_fn = torch.nn.MSELoss()

lr = 0.006   # learning rate

for epoch in range(1000):
    # Forward
    y_pred = forward(x)
    loss = loss_fn(y_pred, y)

    # Backward
    loss.backward()

    # Update (manual SGD)
    with torch.no_grad():
        W -= lr * W.grad
        b -= lr * b.grad

    # Zero gradients
    W.grad.zero_()
    b.grad.zero_()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, W: {W.item():.4f}, b: {b.item():.4f}")


Epoch 0, Loss: 111.8219, W: 0.7646, b: 1.8062
Epoch 100, Loss: 0.1305, W: 3.9999, b: 2.6430
Epoch 200, Loss: 0.0117, W: 4.0000, b: 2.8933
Epoch 300, Loss: 0.0010, W: 4.0000, b: 2.9681
Epoch 400, Loss: 0.0001, W: 4.0000, b: 2.9905
Epoch 500, Loss: 0.0000, W: 4.0000, b: 2.9971
Epoch 600, Loss: 0.0000, W: 4.0000, b: 2.9991
Epoch 700, Loss: 0.0000, W: 4.0000, b: 2.9997
Epoch 800, Loss: 0.0000, W: 4.0000, b: 2.9999
Epoch 900, Loss: 0.0000, W: 4.0000, b: 3.0000


## This is how Gradient Accumulation works

In [ ]:
import torch

W = torch.tensor(2.0, requires_grad=True)

# First pass
y1 = W * 3   # output = 6
y1.backward()
print("Grad after first backward:", W.grad.item())  # 3

# Second pass without zeroing
y2 = W * 4   # output = 8
y2.backward()
print("Grad after second backward:", W.grad.item())  # 3 + 4 = 7


Grad after first backward: 3.0
Grad after second backward: 7.0


## Task 3: Using nn.Module

Re-implement Task 2 using PyTorch’s nn.Module and nn.Linear.

Write a small MyModel class with 1 layer.

Train it using torch.optim.SGD.

👉 Concept learned: modules, optimizers, training loop structure.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Toy dataset
x = torch.linspace(-5, 5, 100).unsqueeze(1)  # (100, 1)
y = 2 * x + 1

class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_1 = nn.Linear(1, 10)  # input_dim=1, output_dim=1
        self.linear_2 = nn.Linear(10, 1)
    def forward(self, x):
        x = F.relu(self.linear_1(x))
        return self.linear_2(x)

model = LinearRegressionModel()

loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.02)


for epoch in range(10000):
    # Forward
    y_pred = model(x)
    loss = loss_fn(y_pred, y)

    # Backward
    optimizer.zero_grad()   # clear old grads
    loss.backward()         # compute new grads
    optimizer.step()        # update params

    if epoch % 100 == 0:
        W, b = model.linear_2.weight.data, model.linear_2.bias.data
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, W: {W}, b: {b}")


Epoch 0, Loss: 30.3852, W: tensor([[-0.1769, -0.2662, -0.3178,  0.0972, -0.3077, -0.1051,  0.4445,  0.1646,
         -0.0528, -0.2795]]), b: tensor([0.3045])
Epoch 100, Loss: 0.0177, W: tensor([[-0.1769, -0.7104, -0.4365,  0.6096, -0.6392, -0.4739,  1.1061,  0.2766,
         -0.0502, -0.4427]]), b: tensor([0.8057])
Epoch 200, Loss: 0.0046, W: tensor([[-0.1769, -0.7246, -0.4048,  0.6521, -0.6166, -0.4864,  1.0586,  0.2302,
         -0.0583, -0.4502]]), b: tensor([0.8484])
Epoch 300, Loss: 0.0022, W: tensor([[-0.1769, -0.7315, -0.3913,  0.6654, -0.6047, -0.4923,  1.0394,  0.2122,
         -0.0617, -0.4537]]), b: tensor([0.8615])
Epoch 400, Loss: 0.0016, W: tensor([[-0.1769, -0.7349, -0.3874,  0.6704, -0.5988, -0.4949,  1.0300,  0.2045,
         -0.0632, -0.4551]]), b: tensor([0.8687])
Epoch 500, Loss: 0.0013, W: tensor([[-0.1769, -0.7369, -0.3878,  0.6722, -0.5956, -0.4961,  1.0247,  0.2011,
         -0.0638, -0.4557]]), b: tensor([0.8737])
Epoch 600, Loss: 0.0011, W: tensor([[-0.1769, -

In [ ]:
loss.item()

39.129642486572266

## TASK 4

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Transform: convert to tensor + normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # mean=0.5, std=0.5
])

# Download and load training + test data
train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


100%|██████████| 9.91M/9.91M [00:01<00:00, 7.21MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.13MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.92MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.04MB/s]


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 128)   # input = 784 → hidden
        self.fc2 = nn.Linear(128, 64)      # hidden → hidden
        self.fc3 = nn.Linear(64, 10)       # hidden → output (10 classes)

    def forward(self, x):
        x = x.view(-1, 28*28)      # flatten image
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)            # no softmax (CrossEntropyLoss expects logits)
        return x


In [ ]:
model = MLP()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.01)

In [ ]:
epoch = 0
for each in range(10):
  model.train()
  for batch_idx,(data,target) in enumerate(train_loader):
    optimizer.zero_grad()
    output = model(data)
    loss = loss_fn(output,target)
    loss.backward()
    optimizer.step()
    epoch+=1

  print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 939, Loss: 0.0896
Epoch 1877, Loss: 0.1118
Epoch 2815, Loss: 0.1095
Epoch 3753, Loss: 0.2776
Epoch 4691, Loss: 0.2378
Epoch 5629, Loss: 0.4568
Epoch 6567, Loss: 0.3984
Epoch 7505, Loss: 0.0149
Epoch 8443, Loss: 0.0159
Epoch 9381, Loss: 0.1825


In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
  for data, target in test_loader:
    output = model(data)
    _, predicted = torch.max(output,1)
    total+= target.size(0)
    correct+= (predicted==target).sum().item()
print(f"Test Accuracy:{100*correct/total:.2f}%")

Test Accuracy:93.96%


## Task 5

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F

# Transform: convert to tensor + normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # mean=0.5, std=0.5
])

# Download and load training + test data
train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 60.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.70MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.9MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.3MB/s]


In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=1,out_channels=16,kernel_size=3,padding=2)
        self.maxpool = nn.MaxPool2d(kernel_size=3) # output from maxpool 16x26x26
        self.fc1 = nn.Linear(in_features=16*10*10, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=10)

    def forward(self, x):
        x = F.relu(self.conv(x))
        x = self.maxpool(x)
        x = torch.flatten(x,1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
model = CNNClassifier()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epoch = 0

for _ in range(10):
  model.train()
  for batch_idx,(data,target) in enumerate(train_loader):
    optimizer.zero_grad()
    output = model(data)
    loss = loss_fn(output,target)
    loss.backward()
    optimizer.step()
    epoch+=1

  print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 939, Loss: 0.1669
Epoch 1877, Loss: 0.0794
Epoch 2815, Loss: 0.0613
Epoch 3753, Loss: 0.0068
Epoch 4691, Loss: 0.0027
Epoch 5629, Loss: 0.0070
Epoch 6567, Loss: 0.0020
Epoch 7505, Loss: 0.0042
Epoch 8443, Loss: 0.1310
Epoch 9381, Loss: 0.0349


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
  for data, target in test_loader:
    output = model(data)
    _, predicted = torch.max(output,1)
    total+= target.size(0)
    correct+= (predicted==target).sum().item()
print(f"Test Accuracy:{100*correct/total:.2f}%")

Test Accuracy:98.57%


## Task 6

---

## Step 1. Understand what a ResNet block does

At its heart, it’s just:

$$
y = F(x) + x
$$

* $F(x)$ = some transformation (usually 2 Conv layers + BatchNorm + ReLU).
* $x$ = the skip connection.
* Then apply ReLU again.

---

## Step 2. Think about the layers inside

A **basic ResNet block** has:

1. `Conv2d(in_channels → out_channels, kernel=3, stride, padding=1)`
2. `BatchNorm2d(out_channels)`
3. ReLU
4. Another `Conv2d(out_channels → out_channels, kernel=3, stride=1, padding=1)`
5. `BatchNorm2d(out_channels)`
6. Skip connection (`x`) added to result.

---

## Step 3. Handle the shortcut

* If input channels = output channels and stride = 1 → shortcut = identity (`x` passes unchanged).
* If not, we need a `1x1 Conv + BatchNorm` to project `x` to the right shape.

---

## Step 4. Let’s write the skeleton together

You start with something like:

```python
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # TODO: define conv1, bn1
        # TODO: define conv2, bn2
        # TODO: define shortcut (identity or 1x1 conv)
        
    def forward(self, x):
        # TODO: implement forward pass
        return out
```

---

💡 Your turn:
👉 Can you try filling in the `__init__` part for `conv1`, `bn1`, `conv2`, and `bn2`?
(Hint: use `nn.Conv2d` with kernel size 3 and padding 1, and `nn.BatchNorm2d`).


In [ ]:
class ResNetBlock(nn.Module):
  def __init__(self,in_channels,out_channels,stride=1):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels=1,out_channels=16,kernel_size=3,stride=1,padding=1)
    self.bn1 = nn.BatchNorm2d()